# 3-1. 하이브리드 · 앙상블 검색

## 학습 목표
- BM25의 점수식(TF, IDF, 문서 길이 정규화)을 설명하고 `rank_bm25`로 구현할 수 있다.
- 한국어 BM25에서 **형태소 분석 토크나이저**(kiwi)가 왜 필수인지 공백 분할과 비교해 보일 수 있다.
- Dense(FAISS) + Sparse(BM25) 결과를 **RRF**로 결합하는 `EnsembleRetriever`를 구성할 수 있다.

## 핵심 키워드
BM25 · IDF · 길이 정규화 · RRF(Reciprocal Rank Fusion) · EnsembleRetriever · Sparse-Dense Hybrid

## 이 노트북의 위치
프로덕션 RAG의 표준 패턴은 **2-stage retrieval** 이다.

```
1단계 (이 노트북)                 2단계 (다음 노트북)
Hybrid (Dense + BM25, RRF)  ─▶  Cross-Encoder Reranker
└─ "넓게 후보를 모은다"            └─ "정답을 꼭대기에 올린다"
       recall 확보                      precision 확보
```

Anthropic의 Contextual Retrieval 실험(2024)도 동일한 구조로 실패율을 **67% 감소**시켰다 (dense 단독 대비). 이 노트북에서는 **1단계(Hybrid)** 를 구성하고, 이어지는 `02_reranking.ipynb` 에서 **2단계(Reranker)** 를 얹어 완성형 파이프라인을 만든다.


In [ ]:
# 공통 세팅: common 헬퍼를 사용하기 위해 repo 루트를 sys.path에 추가
import sys, os
sys.path.insert(0, '../..')

from common import get_chat_model, get_embeddings, provider_badge
from common.ko_tokenizer import tokenize_ko
from common.loaders import load_any

print(provider_badge())


## 1. BM25 점수식 복습

$$\text{score}(q, d) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{f(t,d)\,(k_1+1)}{f(t,d) + k_1 \left(1 - b + b \cdot \frac{|d|}{\text{avgdl}}\right)}$$

- $f(t,d)$: 문서 $d$에서 토큰 $t$의 빈도 (TF)
- $\text{IDF}(t) = \log \frac{N - n_t + 0.5}{n_t + 0.5}$: 희귀 토큰에 더 높은 가중
- $k_1$ (≈ 1.5): TF 포화 제어. 빈도가 커질수록 점수 증가량이 둔화된다.
- $b$ (≈ 0.75): 길이 정규화 강도.

> 💡 **왜 임베딩만 쓰면 안 될까?** 
> 임베딩은 의미를 잘 잡지만 **고유명사·희귀 전문용어**에서는 취약하다 (벡터 공간에서 가까운 이웃이 존재하지 않거나 모호하다). 반면 BM25는 "전자금융감독규정"처럼 exact match가 중요한 키워드 검색에 강하다. 둘을 섞어야 견고하다.


## 2. 한국어 BM25: 토크나이저가 성능을 좌우한다

영어 BM25는 공백 분할만 해도 잘 동작하지만, 한국어는 교착어라서 조사·어미가 붙어 다닌다. "전자금융거래는" ≠ "전자금융거래가" → 단순 공백 분할은 이 둘을 다른 토큰으로 본다.

In [ ]:
# 코퍼스 로드 및 청킹
docs = load_any('../../data/corpus_ko.txt')
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
chunks = splitter.split_documents(docs)
texts = [c.page_content for c in chunks]
print(f'청크 수: {len(texts)}')
print('예시:', texts[0][:80])


In [ ]:
# 두 가지 토크나이저 비교
sample = '전자금융거래는 자동화된 방식으로 이용하는 거래를 말한다.'
print('공백 분할  :', sample.split())
print('kiwi(tokenize_ko):', tokenize_ko(sample))


In [ ]:
from rank_bm25 import BM25Okapi

# (A) 공백 분할 기반 BM25
corpus_ws = [t.split() for t in texts]
bm25_ws = BM25Okapi(corpus_ws)

# (B) kiwi 형태소 분석 기반 BM25
corpus_ko = [tokenize_ko(t) for t in texts]
bm25_ko = BM25Okapi(corpus_ko)

def top_k(bm25, tokenize, query, k=3):
    q = tokenize(query)
    scores = bm25.get_scores(q)
    idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [(scores[i], texts[i][:60]) for i in idx]

query = '전자금융거래가 무엇인지 설명해줘'
print('[공백 분할]')
for s, t in top_k(bm25_ws, str.split, query):
    print(f'  {s:.3f} | {t}')
print('\n[kiwi 형태소]')
for s, t in top_k(bm25_ko, tokenize_ko, query):
    print(f'  {s:.3f} | {t}')


> **관찰**: 공백 분할은 "전자금융거래가"와 "전자금융거래는"을 다른 토큰으로 인식해 매칭이 실패하거나 점수가 낮게 나온다. kiwi는 어간 "전자금융거래"를 동일 토큰으로 추출해 정확히 매칭한다.


## 3. Reciprocal Rank Fusion (RRF)

서로 다른 retriever의 결과(점수 척도가 다름)를 **순위만으로** 결합하는 단순·강건한 기법.

$$\text{RRF}(d) = \sum_{i} \frac{1}{k + \text{rank}_i(d)}$$

- $\text{rank}_i(d)$: retriever $i$가 매긴 문서 $d$의 순위 (1부터 시작)
- $k$ (보통 60): 상위 문서의 영향력을 완화하는 상수

장점:
1. 점수 척도가 달라도(cosine vs BM25) 무관.
2. outlier retriever 하나가 전체를 망치지 않음.
3. 구현이 10줄 이하.


In [ ]:
def rrf_merge(ranked_lists: list[list[str]], k: int = 60) -> list[tuple[str, float]]:
    '''여러 retriever의 결과(문서 ID 리스트)를 RRF로 병합.'''
    scores: dict[str, float] = {}
    for lst in ranked_lists:
        for rank, doc_id in enumerate(lst, start=1):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

# 장난감 예제
dense_top = ['d3', 'd1', 'd5', 'd2']
bm25_top  = ['d1', 'd4', 'd3', 'd6']
print(rrf_merge([dense_top, bm25_top]))


## 4. LangChain EnsembleRetriever로 하이브리드 구성

`EnsembleRetriever`는 내부적으로 RRF를 사용한다. `weights` 인자로 가중합도 지원한다.

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain.retrievers import EnsembleRetriever

# Dense retriever
vectordb = FAISS.from_documents(chunks, get_embeddings())
dense = vectordb.as_retriever(search_kwargs={'k': 5})

# Sparse retriever (kiwi 토크나이저 주입)
sparse = BM25Retriever.from_documents(
    chunks,
    preprocess_func=tokenize_ko,  # 한국어 BM25의 핵심
)
sparse.k = 5

# 앙상블 (RRF 가중치 0.5:0.5)
hybrid = EnsembleRetriever(
    retrievers=[dense, sparse],
    weights=[0.5, 0.5],
)
print('하이브리드 retriever 준비 완료')


In [ ]:
query = '개인정보보호법에서 정의하는 개인정보는?'
print('=== Dense only ===')
for d in dense.invoke(query):
    print('-', d.page_content[:70])
print('\n=== BM25 only ===')
for d in sparse.invoke(query):
    print('-', d.page_content[:70])
print('\n=== Hybrid (RRF) ===')
for d in hybrid.invoke(query):
    print('-', d.page_content[:70])


## 5. 가중치 튜닝 팁

| 상황 | 권장 weights (dense, sparse) |
|------|-----------------------------|
| 일반 자연어 질의 위주 | (0.6, 0.4) |
| 법령·약관·제품코드 등 고유명사 多 | (0.3, 0.7) |
| 짧은 키워드 검색 | (0.2, 0.8) |
| 긴 자연어 의도 질의 | (0.7, 0.3) |

실제로는 RAGAS 평가 프레임워크로 `context_recall` 을 최대화하는 가중치를 탐색한다 (RAGAS 공식 문서를 참고).


## 6. 다음 단계: Reranker 얹기

하이브리드 검색은 **recall을 확보**했지만, 상위 20개 후보의 **순서가 최적이 아닐 수 있다**. BM25와 dense가 각자의 기준으로 매긴 순위를 RRF로 단순 결합한 결과라서, "질문에 정말 답을 주는" 문서가 10위에 묻혀 있을 수 있다.

다음 노트북(`02_reranking.ipynb`)에서는 이 hybrid retriever를 **1단계**로 쓰고, 그 위에 **Cross-Encoder Reranker (BGE-reranker-v2-m3)** 를 2단계로 얹어:

- 후보 top-20 → rerank → top-5 로 정제
- "recall은 이미 확보됐다는 전제에서 precision만 끌어올리기"

이것이 Anthropic Contextual Retrieval 논문이 권장하는 실제 운영 파이프라인이다.